# RL Deslopifier — Live Demo

## What is the Deslopifier?

The **RL Deslopifier** is a fine-tuned language model trained with REINFORCE to rewrite AI-generated text so it reads more like something a human actually wrote. It takes a piece of "sloppy" AI output and produces a targeted rewrite that preserves meaning while reducing the distinctive statistical fingerprints that AI detectors pick up on.

The policy is a LoRA adapter trained on top of **meta-llama/Llama-3.2-3B-Instruct** for 500 gradient steps using a reward signal from EditLens.

## What is EditLens?

**EditLens** (`pangram/editlens_roberta-large`) is a RoBERTa-large model trained to classify text on a 4-bucket scale:

| Label | Meaning |
|-------|----------|
| LABEL_0 | Fully human-written |
| LABEL_1 | Lightly AI-influenced |
| LABEL_2 | Heavily AI-influenced |
| LABEL_3 | Fully AI-generated |

The **EditLens score** is a weighted average of the four label probabilities, mapped to [0, 1] where **0 = fully human** and **1 = fully AI**:

```
score = sum(p_i * i for i in 0..3) / 3.0
```

## What This Notebook Demonstrates

This notebook loads the step-500 checkpoint of the RL Deslopifier and runs it live on example essays, showing EditLens scores before and after rewriting.

### Key Result

| Metric | Baseline (step 0) | Step 500 | Change |
|--------|-------------------|----------|--------|
| Mean EditLens score | 0.999 | 0.670 | **−0.329** |
| % essays in fully-AI zone (LABEL_3) | 100% | 55% | **−45 pp** |
| % essays in human zone (LABEL_0/1) | 0% | 45% | **+45 pp** |

**45% of AI-generated essays were moved out of the fully-AI zone** after a single rewrite pass.

In [ ]:
# Install dependencies
!pip install -q transformers peft datasets accelerate sentencepiece huggingface_hub

In [ ]:
# Log in to HuggingFace (required for gated models like Llama-3.2-3B-Instruct)
from huggingface_hub import login
login()

In [ ]:
# Clone the repo and add scripts/ to the Python path
!git clone https://github.com/braden-j/STAT-4830-PromptLab-project.git

import sys
sys.path.insert(0, '/content/STAT-4830-PromptLab-project/scripts')

## Loading Models

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from fluency_reward_validation import load_editlens, editlens_score

# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Load EditLens reward model ---
print("\nLoading EditLens (pangram/editlens_roberta-large)...")
el_tok, el_mdl = load_editlens(device)
print("EditLens loaded.")

# --- Load Llama-3.2-3B-Instruct + step_500 LoRA adapter ---
BASE_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
LORA_PATH = "/content/STAT-4830-PromptLab-project/outputs/c2_checkpoints/step_500"

print(f"\nLoading base model: {BASE_MODEL_ID}...")
dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
policy_tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if policy_tok.pad_token is None:
    policy_tok.pad_token = policy_tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device.type == "cuda" else None,
)
if device.type == "cpu":
    base_model = base_model.to(device)

print(f"Loading LoRA adapter from: {LORA_PATH}...")
policy_model = PeftModel.from_pretrained(base_model, LORA_PATH)
policy_model.eval()
print("Policy model (Llama-3.2-3B-Instruct + step_500 LoRA) loaded.")

## Helper Functions

In [ ]:
SYSTEM_PROMPT = (
    "You are a skilled human writing editor. Rewrite the following text to sound "
    "more like it was written by a thoughtful human. Make targeted, precise edits. "
    "Preserve the core meaning and approximate length."
)

def score_text(text: str) -> float:
    """Return the EditLens score for text. 0.0 = fully human, 1.0 = fully AI."""
    return editlens_score(el_tok, el_mdl, text, device)


def deslopify(text: str, max_new_tokens: int = 512) -> str:
    """Run text through the RL policy and return the rewritten version."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": text},
    ]
    prompt = policy_tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = policy_tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = policy_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy decoding
            pad_token_id=policy_tok.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return policy_tok.decode(new_ids, skip_special_tokens=True).strip()


print("Helper functions defined: score_text(), deslopify()")

## Live Demo

Three short AI-generated essays on different topics, each rewritten by the RL Deslopifier. EditLens scores are shown before and after.

In [ ]:
EXAMPLES = [
    {
        "topic": "Journalistic",
        "text": (
            "The city council voted unanimously Tuesday to approve a $4.2 million "
            "infrastructure package aimed at addressing longstanding concerns about "
            "road safety in the downtown corridor. The decision, which came after "
            "several months of public comment, represents a significant step forward "
            "in the city's broader commitment to improving transportation outcomes for "
            "all residents. Officials expressed optimism that construction would begin "
            "before the end of the fiscal year."
        ),
    },
    {
        "topic": "Narrative",
        "text": (
            "Elena had always believed that the forest held secrets, but it was not "
            "until her thirty-second birthday that she finally discovered what they "
            "were. As she walked the familiar trail behind her childhood home, a "
            "strange luminescence caught her eye, emanating from a hollow oak she "
            "had climbed hundreds of times as a child. She paused, her breath forming "
            "small clouds in the cold morning air, and felt for the first time in "
            "years that something extraordinary was about to happen."
        ),
    },
    {
        "topic": "Descriptive",
        "text": (
            "The Sonoran Desert is a remarkably diverse ecosystem that spans "
            "approximately 100,000 square miles across the southwestern United States "
            "and northwestern Mexico. Unlike many desert environments, it receives "
            "two distinct rainy seasons, which allows for a striking abundance of "
            "plant and animal life. The iconic saguaro cactus, which can live for "
            "over 150 years and grow to heights exceeding 40 feet, serves as a "
            "symbol of this region's unique capacity to sustain complex life under "
            "conditions that would challenge most organisms."
        ),
    },
]

for i, example in enumerate(EXAMPLES, 1):
    topic = example["topic"]
    original = example["text"]

    print(f"{'='*70}")
    print(f"EXAMPLE {i}: {topic}")
    print(f"{'='*70}")

    print("\nORIGINAL:")
    print(original)

    orig_score = score_text(original)

    print("\nRunning deslopifier...")
    rewrite = deslopify(original)
    rewrite_score = score_text(rewrite)

    print("\nREWRITE:")
    print(rewrite)

    delta = rewrite_score - orig_score
    direction = "▼" if delta < 0 else "▲"
    print(f"\nEditLens score (original):  {orig_score:.4f}  (0=human, 1=AI)")
    print(f"EditLens score (rewrite):   {rewrite_score:.4f}  {direction} {abs(delta):.4f}")
    print()

## Try Your Own Text

Edit `my_text` below with any AI-generated passage and run the cell.

In [ ]:
my_text = "Paste your AI-generated text here."

# --- Score and deslopify ---
print("ORIGINAL:")
print(my_text)
print(f"\nEditLens score (original): {score_text(my_text):.4f}  (0=human, 1=AI)")

print("\nRunning deslopifier...")
my_rewrite = deslopify(my_text)
rewrite_score = score_text(my_rewrite)

print("\nREWRITE:")
print(my_rewrite)
print(f"\nEditLens score (rewrite):  {rewrite_score:.4f}  (0=human, 1=AI)")

## Training Results Summary

### Ternary Progression — REINFORCE C2 Run (500 Steps)

The table below shows how the distribution of EditLens labels shifted across training checkpoints. LABEL_3 = fully AI-generated; LABEL_0/1 = human-like.

| Step | Mean EditLens | LABEL_0 % | LABEL_1 % | LABEL_3 % | KL Divergence |
|------|---------------|-----------|-----------|-----------|---------------|
| 0 (base) | 0.9990 | 0.0 | 0.0 | 100.0 | 0.0000 |
| 50 | 0.7677 | 1.0 | 29.0 | 70.0 | 0.3972 |
| 100 | 0.7664 | 0.0 | 32.0 | 68.0 | 0.4215 |
| 150 | 0.7535 | 0.0 | 36.0 | 64.0 | 0.4567 |
| 200 | 0.7246 | 3.0 | 39.0 | 58.0 | 0.4814 |
| 250 | 0.6819 | 4.0 | 44.0 | 52.0 | 0.5133 |
| 300 | 0.7026 | 5.0 | 35.0 | 60.0 | 0.5495 |
| 350 | 0.7178 | 5.0 | 33.0 | 62.0 | 0.5513 |
| 400 | 0.6749 | 11.0 | 31.0 | 58.0 | 0.5804 |
| 450 | 0.6850 | 9.0 | 34.0 | 56.0 | 0.5447 |
| **500** | **0.6700** | **11.0** | **34.0** | **55.0** | **0.5183** |

**Net change (step 0 → 500):** Mean EditLens −0.329 · LABEL_3 −45 pp · KL +0.518

The KL divergence column measures how far the rewrite distribution has drifted from the base model — a useful guardrail against reward hacking. Values around 0.5 are healthy: the model is meaningfully rewriting while staying fluent.

### E5 Ternary Progression Chart

The chart below visualizes how essay-level EditLens label distributions shift across training steps on a ternary simplex. Each point represents the (LABEL_0, LABEL_1, LABEL_3) proportions at a given checkpoint. Movement toward the LABEL_0/LABEL_1 corner indicates successful human-ification.

In [ ]:
from IPython.display import Image, display
import os

img_path = '/content/STAT-4830-PromptLab-project/outputs/e5_ternary_progression.png'
if os.path.exists(img_path):
    display(Image(img_path))
else:
    print("E5 chart not found. Run scripts/e5_ternary_chart.py first.")